# Imports

In [1]:
import sys
from pathlib import Path
project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.utils import *
from src.hmm import HMM
from src.analysis import *
from src.viterbi import viterbi

import math
import random
from scipy.stats import ttest_ind

# Train HMM

In [2]:
states = ["adapted", "not_adapted"]
filename = "../data/GCF_000001405.40_GRCh38.p14_cds_from_genomic.fna.gz"
hmm = HMM(states)
hmm.initialize_parameters()
hmm.train_emission_probs_from_fasta(fasta_filename=filename)

# Synthetic Data

In [6]:
def generate_sample_sequence(codons, probs, n):
    return "".join(random.choices(codons, weights=probs, k=n))
    

codon_list = generate_all_codons()
probs = [math.exp(hmm.emission_probs["adapted"][codon]) for codon in codon_list]

human_like_seq = {}
random_seq = {}

for i in range(1, 21):
    human_like_seq[f"h{i}"] = generate_sample_sequence(codon_list, probs, 20)
    random_seq[f"r{i}"] = generate_sample_sequence(codon_list, [1/64]*64, 20)

print(human_like_seq)
print(random_seq)

{'h1': 'CAACTTGCAATCCCCCTGAGTGCTGTTGAACTCTCTGGTGATGTGCTGAGTGGATCAAAG', 'h2': 'AGTCACCTCATACTGATCCGCCTGCTTCCTGGGATCCATGATATGCTTTCTGCAATGATG', 'h3': 'CACTCATCCCTACAGGGCGACGTGATGAAGTGGTTATTCCCTCTGGTAAGGGAGGTTTCC', 'h4': 'ATGCCTGGAAAGGCAGACAATCCTGCCGTTACCCAGGCAAAGTTCACTCATACTGTTCCA', 'h5': 'GAAAACTGTTGTGCTGCCAGTGCTCGATCCACTAATTTCCACTACGAGGCTTTGGAATTT', 'h6': 'GAACGGACCTACGGAGTTAAAAAGCCTATCCCCTGCAAACCATATGGGAATAGACGGGTG', 'h7': 'AAGCAACTACAGCACGAGGGGGGACAGGACAGCGCAAGTTCAGATTGGGTTAGACTGGTC', 'h8': 'AGGACTCTTAAGAATGACAAAGCCAAATATGTCGAGGAGCAGAAATTGCAAACAGGGCAA', 'h9': 'ATGACGCAGTGGGCTGTTTTGGCCGGCCTCAAAAAGAAACTGTGGTCCAGTGTCCGATTG', 'h10': 'TGCGGACGTGAGCTGATAGAATACCAAGGGCTCGAATACGCTGGGATTGAATTGAAGAAA', 'h11': 'CGAGAGAAACCGACTGTCCTGCAGCACGTGCCCGGGGGCCTCTATAAGAGCTCGCTGGAC', 'h12': 'GATGACATGAAGAGCGTACCACAGCCCAGCTCCGAGTTTAGCCCTAGAAATCTTCGAGTC', 'h13': 'AAGATTTGTCATGGCTGTACCCAGATCGGCCCTGCTGACGAGTTCTCCGTAGATGAATTC', 'h14': 'CAGCGGAGGCCCCTTGGCCTTTCTTCGAAACATTTCATCTTTCAGAGCGAGCAGGAGTAC', 'h15': 'TTCGGA

# Test with synthetic data

In [7]:
results_human = analyze_genes(human_like_seq, hmm)
results_random = analyze_genes(random_seq, hmm)

human_scores = []
random_scores = []
for score, _ in results_human.values():
    human_scores.append(score)

for score, _ in results_random.values():
    random_scores.append(score)

human_avg = sum(human_scores)/len(human_scores)
random_avg = sum(random_scores)/len(random_scores)

print("Human sequences:")
print(results_human)
print("Random sequences:")
print(results_random)

print(f"Human-like average: {human_avg}")
print(f"Random average: {random_avg}")

Human sequences:
{'h1': (1.0, ['adapted', 'adapted', 'adapted', 'adapted', 'adapted', 'adapted', 'adapted', 'adapted', 'adapted', 'adapted', 'adapted', 'adapted', 'adapted', 'adapted', 'adapted', 'adapted', 'adapted', 'adapted', 'adapted', 'adapted']), 'h2': (1.0, ['adapted', 'adapted', 'adapted', 'adapted', 'adapted', 'adapted', 'adapted', 'adapted', 'adapted', 'adapted', 'adapted', 'adapted', 'adapted', 'adapted', 'adapted', 'adapted', 'adapted', 'adapted', 'adapted', 'adapted']), 'h3': (1.0, ['adapted', 'adapted', 'adapted', 'adapted', 'adapted', 'adapted', 'adapted', 'adapted', 'adapted', 'adapted', 'adapted', 'adapted', 'adapted', 'adapted', 'adapted', 'adapted', 'adapted', 'adapted', 'adapted', 'adapted']), 'h4': (1.0, ['adapted', 'adapted', 'adapted', 'adapted', 'adapted', 'adapted', 'adapted', 'adapted', 'adapted', 'adapted', 'adapted', 'adapted', 'adapted', 'adapted', 'adapted', 'adapted', 'adapted', 'adapted', 'adapted', 'adapted']), 'h5': (1.0, ['adapted', 'adapted', 'adapte

# Statistical test

In [8]:
t_stat, p_val = ttest_ind(human_scores, random_scores)

print("t-stat:", t_stat)
print("p-value:", p_val)

t-stat: 10.21325500993383
p-value: 1.893014922503665e-12
